In [0]:
from pyspark.sql.functions import *

In [0]:
payment=spark.read.format("json")\
    .option("inferSchema", "true")\
    .option("multiLine", "true")\
    .option("mode","PERMISSIVE")\
    .load("s3://retail-lakehouse-ashu/raw/payment/payments.json")

In [0]:
if "_corrupt_record" in payment.columns:
    payment.cache()
    payment.count()

    corrupt_count=payment.filter(col("_corrupt_record").isNotNull()).count()
    print(f"Corrupt Records: {corrupt_count}")
else:
    print("No _corrupt_record column present. JSON parsed successfully.")

In [0]:
payment.printSchema()

In [0]:
payment.show(5)

In [0]:
validation_rule=(
col("amount").isNotNull() &
col("customer_id").isNotNull() &
col("order_id").isNotNull() &
col("payment_id").isNotNull() &
col("payment_method").isNotNull() &
col("payment_timestamp").isNotNull() &
col("refund.eligible").isin([True, False]) &
col("refund.refund_status").isNotNull() &
col("status").isNotNull()
)

In [0]:
good_df=payment.filter(validation_rule)
good_df.display()

In [0]:
bad_df=payment.filter(~validation_rule)
bad_df.display()

In [0]:
bronze_payment=good_df.withColumn("ingestion_timestamp",current_timestamp())\
    .withColumn("source_system",lit("payment"))\
    .withColumn("source_file",col("_metadata.file_path"))\
    .withColumn("batch_id",lit(1))

In [0]:
bronze_payment.display()

In [0]:
total_records=payment.count()

good_records=good_df.count()

bad_records=bad_df.count()

print(f"Total Records   : {total_records}")
print(f"Good Records    : {good_records}")
print(f"Bad Records     : {bad_records}")

In [0]:
duplicate_df=payment.groupBy(col("payment_id")).count().filter(col("count") > 1)

In [0]:
assert total_records==good_records+bad_records

In [0]:
bronze_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://retail-lakehouse-ashu/bronze/payments/")

In [0]:
bad_df=(
    bad_df
        .withColumn("batch_id", lit(1))
        .withColumn("pipeline_name", lit("bronze_payment"))
        .withColumn("source_system", lit("payment"))
        .withColumn("table_name", lit("payment"))
        .withColumn("failure_reason", lit("MANDATORY_FIELD_VALIDATION_FAILED"))
        .withColumn("source_file", col("_metadata.file_path"))   # replace later with metadata
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("raw_record", to_json(struct(*bad_df.columns)))
        .select(
        col("batch_id"),
        col("pipeline_name"),
        col("source_system"),
        col("table_name"),
        col("failure_reason"),
        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_record")
    )
)

In [0]:
bad_df.write \
    .format("delta") \
    .mode("append") \
    .save("s3://retail-lakehouse-ashu/bad_records/")